# US-8: Thorough Data Cleaning Based on Exploration Insights

Clean invalid, nonsensical, and inconsistent records identified during US-7 exploration.

## Prerequisites
- Notebooks 01-06 must have been run to generate the cleaned dataset
- Input: `output/clean_detail_data.parquet` (from US-6)

## Cleaning Steps
| Step | Action | Expected | Rationale |
|------|--------|----------|----------|
| AC1 | Drop SOLD/CLOSED | ~34 rows | Not for sale |
| AC2 | Drop null price (available) | ~22 rows | Core field missing |
| AC3 | Drop owners = 0 | ~25 rows | Impossible for used car |
| AC4 | Drop engine_cap = 0 | ~1 row | Extraction error |
| AC5 | Drop power = 0 | ~1 row | Extraction error |
| AC6 | Drop curb_weight = 2 kg | ~1 row | Extraction error |
| AC7 | Drop manufactured > reg_date year | ~1 row | Temporal impossibility |
| AC8 | Nullify mileage > 500k km | ~5 values | Implausible data |

Expected total: ~85 rows dropped (0.6% data loss)

In [ ]:
import polars as pl
import json
from pathlib import Path
from datetime import datetime

INPUT = Path('output/clean_detail_data.parquet')
EXPORT_PARQUET = Path('output/sgcarmart_used_cars_full.parquet')
EXPORT_CSV = Path('output/sgcarmart_used_cars_full.csv')
REPORT = Path('output/cleaning_report.json')

df = pl.read_parquet(INPUT)
initial_rows = df.shape[0]
print(f'Loaded: {initial_rows:,} rows x {df.shape[1]} columns')
print(f'Columns: {df.columns}')

# Track cleaning steps
steps = []

## Phase 1: Drop Rows

In [ ]:
# AC1: Drop SOLD/CLOSED listings
before = df.shape[0]
dropped_status = df.filter(pl.col('status') != 'Available for sale')
df = df.filter(pl.col('status') == 'Available for sale')
removed = before - df.shape[0]
steps.append({'step': 'AC1: Drop SOLD/CLOSED', 'removed': removed, 'remaining': df.shape[0]})
print(f'AC1: Dropped {removed} SOLD/CLOSED listings ({dropped_status["status"].value_counts()})')
print(f'  Remaining: {df.shape[0]:,}')

In [ ]:
# AC2: Drop rows with null price
before = df.shape[0]
df = df.filter(pl.col('price').is_not_null())
removed = before - df.shape[0]
steps.append({'step': 'AC2: Drop null price', 'removed': removed, 'remaining': df.shape[0]})
print(f'AC2: Dropped {removed} rows with null price')
print(f'  Remaining: {df.shape[0]:,}')

In [ ]:
# AC3: Drop rows with owners = 0
before = df.shape[0]
dropped_owners = df.filter(pl.col('owners') == 0).select(['car_model', 'owners', 'price'])
df = df.filter(pl.col('owners') > 0)
removed = before - df.shape[0]
steps.append({'step': 'AC3: Drop owners=0', 'removed': removed, 'remaining': df.shape[0]})
print(f'AC3: Dropped {removed} rows with owners=0:')
for row in dropped_owners.iter_rows():
    print(f'  {row[0]:45s} owners={row[1]} price=${row[2]:,.0f}')
print(f'  Remaining: {df.shape[0]:,}')

In [ ]:
# AC4: Drop rows with engine_cap = 0
before = df.shape[0]
dropped_ec = df.filter(pl.col('engine_cap') == 0).select('car_model')
df = df.filter((pl.col('engine_cap').is_null()) | (pl.col('engine_cap') > 0))
removed = before - df.shape[0]
steps.append({'step': 'AC4: Drop engine_cap=0', 'removed': removed, 'remaining': df.shape[0]})
print(f'AC4: Dropped {removed} row(s) with engine_cap=0: {dropped_ec["car_model"].to_list()}')
print(f'  Remaining: {df.shape[0]:,}')

In [ ]:
# AC5: Drop rows with power = 0
before = df.shape[0]
dropped_pw = df.filter(pl.col('power') == 0).select('car_model')
df = df.filter((pl.col('power').is_null()) | (pl.col('power') > 0))
removed = before - df.shape[0]
steps.append({'step': 'AC5: Drop power=0', 'removed': removed, 'remaining': df.shape[0]})
print(f'AC5: Dropped {removed} row(s) with power=0: {dropped_pw["car_model"].to_list()}')
print(f'  Remaining: {df.shape[0]:,}')

In [ ]:
# AC6: Drop row with curb_weight = 2 kg (extraction error)
before = df.shape[0]
dropped_cw = df.filter(pl.col('curb_weight') == 2).select('car_model')
df = df.filter((pl.col('curb_weight').is_null()) | (pl.col('curb_weight') != 2))
removed = before - df.shape[0]
steps.append({'step': 'AC6: Drop curb_weight=2', 'removed': removed, 'remaining': df.shape[0]})
print(f'AC6: Dropped {removed} row(s) with curb_weight=2kg: {dropped_cw["car_model"].to_list()}')
print(f'  Remaining: {df.shape[0]:,}')

In [ ]:
# AC7: Drop rows where manufactured > reg_date year
before = df.shape[0]
mismatch = df.filter(
    pl.col('reg_date').is_not_null() &
    pl.col('manufactured').is_not_null() &
    (pl.col('manufactured') > pl.col('reg_date').dt.year())
).select(['car_model', 'manufactured', 'reg_date'])
df = df.filter(
    pl.col('reg_date').is_null() |
    pl.col('manufactured').is_null() |
    (pl.col('manufactured') <= pl.col('reg_date').dt.year())
)
removed = before - df.shape[0]
steps.append({'step': 'AC7: Drop manufactured>reg_date', 'removed': removed, 'remaining': df.shape[0]})
print(f'AC7: Dropped {removed} row(s) where manufactured > reg_date year:')
for row in mismatch.iter_rows():
    print(f'  {row[0]}: manufactured={row[1]}, reg_date={row[2]}')
print(f'  Remaining: {df.shape[0]:,}')

## Phase 2: Nullify Invalid Values

In [ ]:
# AC8: Nullify mileage > 500,000 km
extreme_mileage = df.filter(
    (pl.col('mileage').is_not_null()) & (pl.col('mileage') > 500000)
).select(['car_model', 'mileage'])

nullified_count = extreme_mileage.shape[0]

df = df.with_columns(
    pl.when(pl.col('mileage') > 500000)
    .then(None)
    .otherwise(pl.col('mileage'))
    .alias('mileage')
)

steps.append({'step': 'AC8: Nullify mileage>500k', 'nullified': nullified_count, 'remaining': df.shape[0]})
print(f'AC8: Nullified {nullified_count} mileage values > 500,000 km:')
for row in extreme_mileage.iter_rows():
    print(f'  {row[0]:45s} mileage={row[1]:,} km')
print(f'  Remaining rows: {df.shape[0]:,} (unchanged)')

## Phase 3: Summary & Verification

In [ ]:
# Cleaning summary
total_dropped = initial_rows - df.shape[0]
pct_dropped = total_dropped / initial_rows * 100

print('=' * 60)
print('CLEANING SUMMARY')
print('=' * 60)
print(f'{"Step":40s} {"Removed":>8s} {"Remaining":>10s}')
print('-' * 60)
for s in steps:
    removed = s.get('removed', s.get('nullified', 0))
    print(f'{s["step"]:40s} {removed:>8,} {s["remaining"]:>10,}')
print('-' * 60)
print(f'{"TOTAL":40s} {total_dropped:>8,} {df.shape[0]:>10,}')
print(f'\nData loss: {total_dropped:,} rows ({pct_dropped:.1f}%)')
print(f'Final dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')

In [ ]:
# Verification checks
checks = {}

# No SOLD/CLOSED
non_available = df.filter(pl.col('status') != 'Available for sale').shape[0]
checks['no_sold_closed'] = non_available == 0

# No null price
null_price = df['price'].null_count()
checks['no_null_price'] = null_price == 0

# No owners = 0
zero_owners = df.filter(pl.col('owners') == 0).shape[0]
checks['no_zero_owners'] = zero_owners == 0

# No engine_cap = 0
zero_ec = df.filter(pl.col('engine_cap') == 0).shape[0]
checks['no_zero_engine_cap'] = zero_ec == 0

# No power = 0
zero_pw = df.filter(pl.col('power') == 0).shape[0]
checks['no_zero_power'] = zero_pw == 0

# No curb_weight = 2
cw_2 = df.filter(pl.col('curb_weight') == 2).shape[0]
checks['no_curb_weight_2'] = cw_2 == 0

# No manufactured > reg_date year
manuf_mismatch = df.filter(
    pl.col('reg_date').is_not_null() &
    pl.col('manufactured').is_not_null() &
    (pl.col('manufactured') > pl.col('reg_date').dt.year())
).shape[0]
checks['no_manuf_gt_regdate'] = manuf_mismatch == 0

# No mileage > 500k
extreme_mile = df.filter(pl.col('mileage') > 500000).shape[0]
checks['no_extreme_mileage'] = extreme_mile == 0

# All IDs unique
dupes = df.shape[0] - df.select(pl.col('listing_id').n_unique()).item()
checks['unique_ids'] = dupes == 0

print('=== Verification Checks ===')
all_pass = True
for check, passed in checks.items():
    status = 'PASS' if passed else 'FAIL'
    if not passed:
        all_pass = False
    print(f'  [{status}] {check}')

print(f'\n{"ALL CHECKS PASSED" if all_pass else "SOME CHECKS FAILED"}')

## Phase 4: Export

In [ ]:
# Export clean Parquet (overwrite US-7 exports)
cat_cols = [c for c in df.columns if df[c].dtype == pl.Categorical]
df_export = df.with_columns([pl.col(c).cast(pl.Utf8) for c in cat_cols])

df_export.write_parquet(EXPORT_PARQUET, compression='snappy')
df_export.write_csv(EXPORT_CSV)

print(f'Exported: {EXPORT_PARQUET} ({EXPORT_PARQUET.stat().st_size / 1024 / 1024:.1f} MB)')
print(f'Exported: {EXPORT_CSV} ({EXPORT_CSV.stat().st_size / 1024 / 1024:.1f} MB)')

# Verify reload
df_verify = pl.read_parquet(EXPORT_PARQUET)
print(f'Verification: {df_verify.shape[0]:,} rows x {df_verify.shape[1]} columns')

In [ ]:
# Export cleaning report
report = {
    'generated_at': datetime.now().isoformat(),
    'source_file': str(INPUT),
    'rows_before': initial_rows,
    'rows_after': df.shape[0],
    'rows_removed': initial_rows - df.shape[0],
    'pct_removed': round((initial_rows - df.shape[0]) / initial_rows * 100, 1),
    'steps': steps,
    'verification': {k: v for k, v in checks.items()},
}

with open(REPORT, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Cleaning report: {REPORT}')
print(f"\n{'=' * 60}")
print('US-8: THOROUGH DATA CLEANING COMPLETE')
print(f"{'=' * 60}")
print(f'  Before: {initial_rows:,} rows')
print(f'  After:  {df.shape[0]:,} rows')
print(f'  Removed: {initial_rows - df.shape[0]:,} rows ({(initial_rows - df.shape[0]) / initial_rows * 100:.1f}%)')